In [432]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
import warnings
warnings.filterwarnings('ignore')

In [433]:
data = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"])

In [434]:
data

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [435]:
data.shape

(5572, 2)

In [436]:
data['label'].unique()

array(['ham', 'spam'], dtype=object)

In [437]:
data['label'].value_counts()

,count
label,
ham,4825
spam,747


In [438]:
data = data.drop_duplicates()
data

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [439]:
data.shape

(5169, 2)

In [440]:
data.isnull().sum()

,0
label,0
message,0


Converting Label (ham , spam) into (1 , 0)

In [441]:
data['label'] = data['label'].map({'ham' : 1,'spam' : 0})

In [442]:
data.head()

,label,message
0,1,"Go until jurong point, crazy.. Available only ..."
1,1,Ok lar... Joking wif u oni...
2,0,Free entry in 2 a wkly comp to win FA Cup fina...
3,1,U dun say so early hor... U c already then say...
4,1,"Nah I don't think he goes to usf, he lives aro..."


Convert all messages into lower case

In [443]:
data['message'] = data['message'].apply(lambda x: x.lower())

Removing Punctuation from message

In [444]:
def remove_punctuation(message):
  return "".join([i for i in message if i not in string.punctuation])


In [445]:
data['message'] = data['message'].apply(remove_punctuation)

Removing numbers from message

In [446]:
def remove_numbers(message):
  return "".join([i for i in message if not i.isdigit()])

In [447]:
data['message'] = data['message'].apply(remove_numbers)

Remove URL from message


In [448]:
def remove_url(message):
  message = re.sub(r'http\S+', '', message)
  return message

In [449]:
data['message'] = data['message'].apply(remove_url)

Remove emojis from message

In [450]:
def remove_emojis(message):
   new = ""
   for i in message:
       if i.isascii():
         new += i
   return new
data['message'] = data['message'].apply(remove_emojis)

In [451]:
data

,label,message
0,1,go until jurong point crazy available only in ...
1,1,ok lar joking wif u oni
2,0,free entry in a wkly comp to win fa cup final...
3,1,u dun say so early hor u c already then say
4,1,nah i dont think he goes to usf he lives aroun...
...,...,...
5567,0,this is the nd time we have tried contact u u...
5568,1,will b going to esplanade fr home
5569,1,pity was in mood for that soany other suggest...
5570,1,the guy did some bitching but i acted like id ...


Stopwords

In [452]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [453]:
def remove_sw(text):
  words = nltk.word_tokenize(text)
  cleaned = []
  for i in words:
    if i not in stop_words:
      cleaned.append(i)
  return ' '.join(cleaned)

In [454]:
data['message'] = data['message'].apply(remove_sw)

In [455]:
data

,label,message
0,1,go jurong point crazy available bugis n great ...
1,1,ok lar joking wif u oni
2,0,free entry wkly comp win fa cup final tkts st ...
3,1,u dun say early hor u c already say
4,1,nah dont think goes usf lives around though
...,...,...
5567,0,nd time tried contact u u pound prize claim ea...
5568,1,b going esplanade fr home
5569,1,pity mood soany suggestions
5570,1,guy bitching acted like id interested buying s...


In [456]:
X = data['message']
y = data['label']

In [457]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42)

In [458]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

Naive Bayes

In [459]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
model_nb = MultinomialNB()
model_nb.fit(X_train,y_train)


MultinomialNB()

In [460]:
y_pred_nb = model_nb.predict(X_test)

In [461]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print("Accuracy:", accuracy_nb)

Accuracy: 0.97678916827853


Logistic Regression

In [462]:
from sklearn.linear_model import LogisticRegression
model_lr = LogisticRegression()
model_lr.fit(X_train,y_train)

LogisticRegression()

In [463]:
y_pred_lr  = model_lr.predict(X_test)

In [464]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("Accuracy:", accuracy_lr)

Accuracy: 0.97678916827853


SVM

In [465]:
from sklearn.svm import SVC
model_svm = SVC()
model_svm.fit(X_train,y_train)

SVC()

In [466]:
y_pred_svm = model_svm.predict(X_test)

In [467]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("Accuracy:", accuracy_svm)

Accuracy: 0.971953578336557


In [468]:
import joblib
joblib.dump(model_nb,'naive_bayes_model.pkl')
joblib.dump(vectorizer,'vectorizer.pkl')


['vectorizer.pkl']